# 🏭 Planificación de Producción — Optimización con HIGHS (vía SciPy)

**Problema:** Una fábrica produce 3 productos (A, B, C) en 3 turnos (Mañana, Tarde, Noche).
Cada turno tiene un límite de horas disponibles y cada producto consume recursos diferentes.
El objetivo es **maximizar las ganancias totales** respetando las restricciones de capacidad.

> ⚡ **Nota para JupyterLite:** Este notebook usa `scipy.optimize.linprog` con el método `highs`.
> SciPy incluye el solver **HiGHS** compilado internamente, por lo que funciona directamente
> en el navegador (JupyterLite/Pyodide) **sin necesidad de instalar `pyomo` ni `highspy`**
> (que no están disponibles en el entorno WASM).

## Modelo matemático

$$\text{Maximizar: } Z = \sum_{p \in \text{productos}} \sum_{t \in \text{turnos}} \text{ganancia}_{p,t} \cdot x_{p,t}$$

Sujeto a:

$$x_{p,t} \geq 0 \quad \forall p, t$$

$$\sum_{p \in \text{productos}} \text{horas}_{p,t} \cdot x_{p,t} \leq \text{capacidad}_t \quad \forall t$$

$$\sum_{t \in \text{turnos}} x_{p,t} \leq \text{demanda}_p \quad \forall p$$

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# En JupyterLite, instalamos paquetes con piplite (solo los disponibles en Pyodide)
try:
    import piplite
    await piplite.install(['scipy'])
    print("📦 scipy instalado vía piplite (JupyterLite)")
except ImportError:
    print("📦 Entorno local: scipy ya disponible")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

print("✅ Bibliotecas cargadas correctamente")

## 📋 Datos del problema

In [ ]:
# Productos y sus ganancias por unidad en cada turno
productos = ['Producto A', 'Producto B', 'Producto C']
turnos = ['Mañana', 'Tarde', 'Noche']

# Ganancia por unidad de cada producto en cada turno (en miles de USD)
ganancias = {
    'Producto A': {'Mañana': 12, 'Tarde': 10, 'Noche': 8},
    'Producto B': {'Mañana': 15, 'Tarde': 13, 'Noche': 11},
    'Producto C': {'Mañana': 9,  'Tarde': 7,  'Noche': 6},
}

# Horas que consume cada producto en cada turno
horas_consumo = {
    'Producto A': {'Mañana': 2.0, 'Tarde': 1.5, 'Noche': 1.0},
    'Producto B': {'Mañana': 3.0, 'Tarde': 2.5, 'Noche': 2.0},
    'Producto C': {'Mañana': 1.5, 'Tarde': 1.0, 'Noche': 0.8},
}

# Capacidad máxima de horas por turno
capacidad_turnos = {'Mañana': 100, 'Tarde': 80, 'Noche': 60}

# Demanda máxima de cada producto
demanda_maxima = {'Producto A': 30, 'Producto B': 25, 'Producto C': 40}

# Mostrar datos en tablas
df_ganancias = pd.DataFrame(ganancias).T
df_horas = pd.DataFrame(horas_consumo).T
df_demanda = pd.DataFrame(list(demanda_maxima.items()), columns=['Producto', 'Demanda Máx.']).set_index('Producto')

print("📊 Ganancia por unidad (miles USD):")
display(df_ganancias)

print("\n⏱️ Horas consumidas por unidad:")
display(df_horas)

print("\n🏗️ Capacidad por turno (horas):")
display(pd.DataFrame({'Capacidad': capacidad_turnos}))

print("\n📦 Demanda máxima por producto:")
display(df_demanda)

## 🔧 Construcción del modelo con SciPy

`scipy.optimize.linprog` resuelve problemas de **minimización** de la forma:

$$\min c^T x \quad \text{sujeto a } A_{ub} x \leq b_{ub}, \; A_{eq} x = b_{eq}, \; x \geq 0$$

Como nuestro problema es de **maximización**, multiplicamos la función objetivo por **-1**.

Las variables se ordenan como: `x = [A_M, A_T, A_N, B_M, B_T, B_N, C_M, C_T, C_N]`
(producto × turno).

In [ ]:
# Orden de las variables: (producto, turno)
indices = [(p, t) for p in productos for t in turnos]
n_vars = len(indices)

# Coeficientes de la función objetivo (negativos porque linprog minimiza)
c = np.array([-ganancias[p][t] for p, t in indices], dtype=float)

# --- Restricciones de capacidad por turno (A_ub x <= b_ub) ---
A_capacidad = np.zeros((len(turnos), n_vars))
for i, t in enumerate(turnos):
    for j, (p, tj) in enumerate(indices):
        if tj == t:
            A_capacidad[i, j] = horas_consumo[p][t]
b_capacidad = np.array([capacidad_turnos[t] for t in turnos], dtype=float)

# --- Restricciones de demanda por producto (A_ub x <= b_ub) ---
A_demanda = np.zeros((len(productos), n_vars))
for i, p in enumerate(productos):
    for j, (pj, t) in enumerate(indices):
        if pj == p:
            A_demanda[i, j] = 1.0
b_demanda = np.array([demanda_maxima[p] for p in productos], dtype=float)

# Combinar restricciones
A_ub = np.vstack([A_capacidad, A_demanda])
b_ub = np.concatenate([b_capacidad, b_demanda])

print("✅ Matrices construidas")
print(f"   Variables: {n_vars}")
print(f"   Restricciones: {len(b_ub)}")

## ⚙️ Resolución con HIGHS (método `highs` de SciPy)

In [ ]:
# Resolver con el solver HiGHS (compilado dentro de SciPy)
result = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    bounds=(0, None),
    method='highs',
)

print(f"📌 Estado del solver: {result.status} ({result.message})")
print(f"📌 ¿Óptimo encontrado?: {result.success}")

if result.success:
    x_opt = result.x
    ganancia_total = -result.fun  # recuperamos el signo (era maximización)
    print(f"\n💰 GANANCIA TOTAL: ${ganancia_total:,.2f} miles USD")
else:
    raise RuntimeError("El solver no encontró una solución óptima.")

## 📊 Análisis de resultados

In [ ]:
# Extraer resultados en DataFrame
resultados = []
for j, (p, t) in enumerate(indices):
    val = x_opt[j]
    if val > 0.001:  # Solo mostrar producciones significativas
        resultados.append({
            'Producto': p,
            'Turno': t,
            'Cantidad': round(val, 2),
            'Ganancia Unit.': ganancias[p][t],
            'Ganancia Total': round(ganancias[p][t] * val, 2),
        })

df_resultados = pd.DataFrame(resultados)
print("📋 Plan de producción óptimo:")
display(df_resultados)

# Horas utilizadas por turno
horas_usadas = {}
for t in turnos:
    h = sum(
        horas_consumo[p][t] * x_opt[j]
        for j, (p, tj) in enumerate(indices) if tj == t
    )
    horas_usadas[t] = round(h, 2)

df_horas_usadas = pd.DataFrame({
    'Horas Usadas': horas_usadas,
    'Capacidad': capacidad_turnos,
    '% Utilizado': [round(horas_usadas[t] / capacidad_turnos[t] * 100, 1) for t in turnos]
}).T
print("\n⏱️ Uso de capacidad por turno:")
display(df_horas_usadas)

## 📈 Visualización de resultados

In [ ]:
# Configuración de estilo
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📊 Análisis de Planificación de Producción Óptima', fontsize=14, fontweight='bold', y=1.02)
colors = {'Mañana': '#2196F3', 'Tarde': '#FF9800', 'Noche': '#9C27B0'}
product_colors = {'Producto A': '#4CAF50', 'Producto B': '#F44336', 'Producto C': '#2196F3'}

# Gráfico 1: Producción por producto y turno
ax1 = axes[0, 0]
x_pos = np.arange(len(productos))
width = 0.25
for i, t in enumerate(turnos):
    vals = [x_opt[indices.index((p, t))] for p in productos]
    bars = ax1.bar(x_pos + i * width, vals, width, label=t, color=colors[t], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9)
ax1.set_xlabel('Producto', fontsize=11)
ax1.set_ylabel('Cantidad producida', fontsize=11)
ax1.set_title('Producción por Producto y Turno', fontsize=12, fontweight='bold')
ax1.set_xticks(x_pos + width)
ax1.set_xticklabels(productos)
ax1.legend(fontsize=9)

# Gráfico 2: Ganancia por producto
ax2 = axes[0, 1]
ganancias_prod = [sum(ganancias[p][t] * x_opt[indices.index((p, t))] for t in turnos) for p in productos]
bars = ax2.bar(productos, ganancias_prod, color=[product_colors[p] for p in productos], alpha=0.85)
for bar, val in zip(bars, ganancias_prod):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'${val:.1f}k', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('Ganancia (miles USD)', fontsize=11)
ax2.set_title('Ganancia Total por Producto', fontsize=12, fontweight='bold')

# Gráfico 3: Uso de capacidad por turno
ax3 = axes[1, 0]
cap_pct = [horas_usadas[t] / capacidad_turnos[t] * 100 for t in turnos]
bars = ax3.bar(turnos, cap_pct, color=[colors[t] for t in turnos], alpha=0.85)
for bar, val in zip(bars, cap_pct):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax3.set_ylabel('Uso (%)', fontsize=11)
ax3.set_title('Uso de Capacidad por Turno', fontsize=12, fontweight='bold')
ax3.set_ylim(0, 110)
ax3.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='Límite')
ax3.legend(fontsize=9)

# Gráfico 4: Donut de contribución a la ganancia
ax4 = axes[1, 1]
labels = [f'{p}\n${g:.1f}k' for p, g in zip(productos, ganancias_prod)]
wedges, texts, autotexts = ax4.pie(
    ganancias_prod, labels=labels, colors=[product_colors[p] for p in productos],
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    textprops={'fontsize': 9}
)
ax4.set_title('Contribución a Ganancia Total', fontsize=12, fontweight='bold')
centre_circle = plt.Circle((0, 0), 0.55, fc='white')
ax4.add_artist(centre_circle)
ax4.text(0, 0, f'${ganancia_total:.1f}k', ha='center', va='center',
         fontsize=13, fontweight='bold', color='#333')

plt.tight_layout()
plt.show()
print("📊 Gráficos mostrados correctamente")

## 📝 Análisis y conclusiones

In [ ]:
print("=" * 60)
print("📋 RESUMEN EJECUTIVO")
print("=" * 60)
print(f"\n💰 Ganancia total óptima: ${ganancia_total:,.2f} miles USD")
print(f"\n📦 Producción total: {sum(df_resultados['Cantidad']):.1f} unidades")
print(f"\n🏆 Producto más rentable: {df_resultados.loc[df_resultados['Ganancia Total'].idxmax(), 'Producto']}")
print(f"   Ganancia del mejor producto: ${df_resultados['Ganancia Total'].max():.2f}k")

turno_mas_usado = max(capacidad_turnos, key=lambda t: horas_usadas[t] / capacidad_turnos[t])
print(f"\n⚠️ Turno más crítico: {turno_mas_usado} ({cap_pct[turnos.index(turno_mas_usado)]:.1f}% usado)")
print(f"   Si se pudieran agregar {capacidad_turnos[turno_mas_usado] - horas_usadas[turno_mas_usado]:.1f} horas,")
print(f"   se podría expandir la producción.")

print("\n" + "=" * 60)
print("💡 INSIGHTS:")
print("=" * 60)

# Producto con mayor ganancia por hora de recurso
gan_hora = {}
for p in productos:
    max_gh = max(ganancias[p][t] / horas_consumo[p][t] for t in turnos)
    gan_hora[p] = max_gh
    print(f"   {p}: ${max_gh:.2f} de ganancia por hora de recurso")

mejor_producto = max(gan_hora, key=gan_hora.get)
print(f"\n   ✅ {mejor_producto} tiene la mejor relación ganancia/hora.")
print(f"   💡 Estrategia: priorizar {mejor_producto} en los turnos con mayor ganancia.")

print("\n" + "=" * 60)